In [ ]:
import os
from pathlib import Path

# Get the current notebook's directory
notebook_dir = Path().absolute()

# Get the parent directory of the current notebook's folder
parent_dir = notebook_dir.parent

# Change the working directory to the parent folder
os.chdir(parent_dir)

# Print to verify the change
print("New Working Directory:", os.getcwd())

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from rtree import index
from tqdm import tqdm

tqdm.pandas()

# === STEP 1: Caricamento dati ===

# Dataset popolazione da redistribuire
df_pop = pd.read_csv("dati_istat_pop_1km2.csv")
gdf_pop = gpd.GeoDataFrame(df_pop, geometry=gpd.points_from_xy(df_pop['longitude'], df_pop['latitude']), crs="EPSG:4326")

# Dataset centroidi con COD_MACRO
df_zone = pd.read_csv("DATASET_DIST/centroidi_con_zone.csv")
gdf_zone = gpd.GeoDataFrame(df_zone, geometry=gpd.points_from_xy(df_zone['longitudine'], df_zone['latitudine']), crs="EPSG:4326")

# === STEP 2: Dizionario dei pesi associati alle zone ===
pesatura_zona = {
    'M01': 0.35,   # Area residenziale
    'M02': 0.10,   # Servizi pubblici
    'M03': 0.09,   # Impianti industriali
    'M04': 0.07,   # Trasporti e reti di comunicazione
    'M05': 0.10,   # Turismo e attività ricreative
    'M06': 0.05,   # Agricoltura
    'M07': 0.05,   # Trattamento delle acque e rifiuti
    'M08': 0.03,   # Aree naturali
    'M09': 0.09,   # Luoghi di interesse culturale
    'M10': 0.0,    # Corpi idrici e zone umide
    'M99': 0.09    # Altro
}

# === STEP 3: Crea indice spaziale su gdf_zone ===
def crea_indice_spaziale(df):
    idx = index.Index()
    for i, row in tqdm(df.iterrows(), total=len(df), desc="🔧 Creazione indice spaziale"):
        point = row.geometry
        idx.insert(i, (point.x, point.y, point.x, point.y))
    return idx

indice_zone = crea_indice_spaziale(gdf_zone)

# === STEP 4: Redistribuisci popolazione ===

# Lista variabili da redistribuire
variabili_pop = ['TOT_P_2021', 'M', 'F', 'Y_LT15', 'Y1564', 'Y_GE65', 
                 'NAT', 'EU_OTH', 'OTH', 'EMP', 'SAME', 'CHG_IN', 'CHG_OUT']

# Inizializza dizionario aggregato
dati_aggregati = {i: {var: 0 for var in variabili_pop} for i in gdf_zone.index}

# Funzione per distribuire una riga su 4 vicini
def distribuisci_su_vicini(row):
    point = row.geometry
    vicini_idx = list(indice_zone.nearest((point.x, point.y, point.x, point.y), 4))
    cod_macro = gdf_zone.loc[vicini_idx, 'COD_MACRO']
    pesi = [pesatura_zona.get(c, 1.0) for c in cod_macro]
    somma_pesi = sum(pesi)

    if somma_pesi == 0:
        return

    for idx_vicino, peso in zip(vicini_idx, pesi):
        frazione = peso / somma_pesi
        for var in variabili_pop:
            dati_aggregati[idx_vicino][var] += row[var] * frazione

# Applica la redistribuzione
gdf_pop.progress_apply(distribuisci_su_vicini, axis=1)

# === STEP 5: Assegna i dati aggregati a gdf_zone ===

for var in variabili_pop:
    gdf_zone[var] = gdf_zone.index.map(lambda i: round(dati_aggregati[i][var], 2))

# === STEP 6: Prepara lat/long e salva ===

gdf_zone['latitudine'] = gdf_zone.geometry.y
gdf_zone['longitudine'] = gdf_zone.geometry.x

# Salva CSV con le variabili redistribuite
output_cols = ['latitudine', 'longitudine', 'COD_MACRO'] + variabili_pop
gdf_zone[output_cols].to_csv("DATASET_DIST/popolazione_redistribuita.csv", index=False)

# Salva GeoPackage con le variabili redistribuite
gdf_zone.to_file("DATASET_DIST/popolazione_redistribuita.gpkg", driver="GPKG")

print("✅ File CSV e GPKG creati con successo!")
